# Sentiment Analysis API – Twitter Airline Tweets

## Overview

This tutorial describes the internal Python API for a simple sentiment
analysis pipeline on social media posts (tweets). The goal of the API is:

- to make it easy to plug in a CSV file with **text + label**,
- to run **profiling + preprocessing + training + evaluation** with a few
  function calls, and
- to expose a **`predict_sentiment`** function that can be reused in other
  projects.

Under the hood we use:

- **pandas** for data handling  
- **ydata-profiling** for automatic EDA  
- **scikit-learn** for TF-IDF vectorization and a Logistic Regression
  classifier  

The public API lives in a single module:

```text
sentiment_utils.py


In [1]:
import pandas as pd

import importlib
import sentiment_utils
importlib.reload(sentiment_utils)
from sentiment_utils import (
    load_data,
    preprocess_dataframe,
    split_data,
    vectorize_and_train,
    evaluate_model,
    predict_sentiment,
    LABEL_MAP,
)


DATA_PATH = "data/Tweets.csv"


## Load and preprocess the data

We load the raw CSV, clean the text (lowercasing, removing URLs, usernames,
extra spaces, etc.), and create a numeric `label` column:

- `0` → negative  
- `1` → neutral  
- `2` → positive  

Then we quickly inspect the label distribution.


In [2]:
# === Load raw data ===
df_raw = load_data(DATA_PATH)
print("Raw shape:", df_raw.shape)

# === Clean text + build label column ===
df_clean = preprocess_dataframe(df_raw)
print("After cleaning:", df_clean.shape)

print("\nLabel distribution:")
print(
    df_clean["label"]
    .value_counts()
    .sort_index()
    .rename(index={0: "negative", 1: "neutral", 2: "positive"})
)


Raw shape: (14640, 2)
After cleaning: (14640, 3)

Label distribution:
label
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


## Load trained model and vectorizer

Here we load the TF–IDF vectorizer and logistic regression model that were
trained earlier in `sentiment.example.ipynb` and saved to disk. These
objects will be reused to serve predictions without retraining.


In [3]:
from joblib import load

VECT_PATH = "tfidf_vectorizer.joblib"
MODEL_PATH = "logreg_sentiment_model.joblib"

vectorizer = load(VECT_PATH)
model = load(MODEL_PATH)

print("Vectorizer loaded from:", VECT_PATH)
print("Model loaded from:", MODEL_PATH)


Vectorizer loaded from: tfidf_vectorizer.joblib
Model loaded from: logreg_sentiment_model.joblib


## API-style helper for single-text prediction

We define a thin wrapper function that uses the loaded vectorizer and model
to predict the sentiment label for a single piece of text. This imitates
what an API endpoint would do.


In [8]:
def predict_sentiment_api(text: str) -> str:
    """
    Predict sentiment for a single text string.

    Parameters
    ----------
    text : str
        Input tweet / review text.

    Returns
    -------
    label : str
        One of {"negative", "neutral", "positive"}.
    """
    # predict_sentiment returns a list, so take the first element
    return predict_sentiment(text, vectorizer, model)[0]

examples = [
    "I love this airline, great service and friendly staff!",
    "This was the worst flight of my life. Delayed and rude crew.",
    "The flight was on time and everything went as expected.",
]

for t in examples:
    print(predict_sentiment_api(t), " | ", t)


positive  |  I love this airline, great service and friendly staff!
negative  |  This was the worst flight of my life. Delayed and rude crew.
positive  |  The flight was on time and everything went as expected.


In [9]:
def show_probs(text):
    X = vectorizer.transform([text])
    probs = model.predict_proba(X)[0]
    print(text)
    for cls, p in zip(model.classes_, probs):
        print(f"  {LABEL_MAP[int(cls)]}: {p:.3f}")

show_probs("The flight was okay, nothing special but nothing terrible.")

The flight was okay, nothing special but nothing terrible.
  negative: 0.885
  neutral: 0.102
  positive: 0.013
